# Data Analysis Agent (with memory)
Run the cells **top to bottom**, one at a time. You need a free Gemini API key saved as a Colab secret named `GEMINI_API_KEY` (key icon in the left sidebar, then switch on Notebook access).

## 1. Install libraries

In [ ]:
!pip -q install google-genai pandas matplotlib openpyxl

## 2. Download the project from GitHub

In [ ]:
import os, subprocess
REPO = "https://github.com/beingbharatiya7/Data-analysis-agent.git"
if not os.path.exists("Data-analysis-agent"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("Data-analysis-agent")
print(os.listdir("."))

## 3. Load your API key (saved earlier in Colab Secrets)

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("API key loaded:", bool(os.environ["GEMINI_API_KEY"]))

## 4. Test the tools and the memory (no AI needed)

In [ ]:
!python test_tools.py > /dev/null && echo "Tools test: OK"
!python test_memory.py

## 5. Create the agent, and a small helper to show answers nicely

In [ ]:
from agent import DataAgent
from IPython.display import Markdown, Image, display

agent = DataAgent()
print(agent.load("sample_data/store_sales.csv"))

def ask(question):
    answer = agent.ask(question)
    display(Markdown(answer))
    for t in agent.trace:
        if t["tool"] == "plot_chart" and "saved_to" in t["result"]:
            display(Image(t["result"]["saved_to"]))

## 6. Analysis questions (the `[tool]` lines show the real tool calls)

In [ ]:
ask("Which region has the highest average total_sales, and how strong is the link between units and total_sales?")

In [ ]:
ask("Plot the distribution of total_sales.")

## 7. MEMORY demo - teach the agent a definition

In [ ]:
ask("Remember this: 'big orders' means total_sales greater than 1000.")

Now ask something that needs that definition. Look for the `[memory]` line and for `filter_rows` in the tool calls.

In [ ]:
ask("How many big orders are there and what percent of all orders are they?")

## 8. Simulate a NEW session (a brand new agent object - the memory must still be there)

In [ ]:
agent = DataAgent()
agent.load("sample_data/store_sales.csv")
ask("Which region has the most big orders?")

## 9. Inspect what is remembered

In [ ]:
print(agent.memory.pretty())

## 10. Correct a fact (the old value gets replaced, not duplicated)

In [ ]:
ask("Actually, big orders should mean total_sales greater than 1500. Please update that.")

In [ ]:
print(agent.memory.pretty())

## 11. Delete a memory

In [ ]:
ask("Forget the big orders note.")
print(agent.memory.pretty())

## 12. Try another dataset and your own question

In [ ]:
print(agent.load("sample_data/student_performance.csv"))
ask("Does study_hours affect exam_score? Which branch scores best on average?")

In [ ]:
q = input("Your question: ")
ask(q)